# FuzzyDiff: SDXL generation and spatially guided attention refinement

Run top to bottom in Kaggle with **Internet enabled** and a **T4 GPU**.
The notebook uses one GPU with CPU offload; T4 x2 memory is not pooled.
Start with `SMOKE_TEST = True` in the final cell, then set it to `False`
for the 768 x 768, 50-step run. Outputs and run metadata go to
`/kaggle/working/outputs` (or `./outputs` locally).

Each tracked phrase becomes a spatial membership function with values in `[0, 1]`,
built from the phrase's share of the per-position cross-attention distribution.
The guidance loss evaluates phrase presence and explicitly supplied 2D relations
on those maps as graded truth values. Hedge intensity ("slightly", "very") remains
the pretrained model's interpretation; this is not a calibrated intensity predictor.
CLIP similarity is a diagnostic, not proof of visual or relational quality.

For image-quality diagnosis, run `python run_fullpipeline.py --compare` to save
native SDXL, custom guidance-off and guidance-on results for the same
prompt, seed and settings. This comparison skips refinement.


In [ ]:
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import List, Dict, Optional, Union
import gc
import json
import math
import warnings
import torch
from PIL import Image
from torch.nn.functional import interpolate
import torch.nn.functional as F
import numpy as np


In [ ]:
# Keep Kaggle's CUDA-enabled torch/torchvision installation.
%pip install -q "diffusers==0.35.1" "transformers==4.56.2" "accelerate==1.10.1" "safetensors>=0.4.3" "git+https://github.com/openai/CLIP.git"
import clip


## Configuration

In [ ]:
@dataclass
class RunConfig:
    prompt: str = "a black dog and a white cat"
    seeds: List[int] = field(default_factory=lambda: [84])
    output_path: Path = field(default_factory=lambda: Path('/kaggle/working/outputs')
                              if Path('/kaggle/working').exists() else Path('./outputs'))
    n_inference_steps: int = 50
    guidance_scale: float = 9.5
    height: int = 768
    width: int = 768
    attn_res: int = 16
    max_iter_to_alter: int = 25
    attend_excite_lr: float = 0.2
    alpha: float = 10.0
    spatial_loss_weight: float = 0.5
    # Sharpness of the per-position token distribution that defines membership.
    # High values drive membership toward 0/1; lower values keep truths graded.
    membership_sharpness: float = 100.0
    # Relative mode measures spatial support, not calibrated visual presence.
    membership_mode: str = 'share'
    # Fuzzy conjunction over phrases. 'min' (Goedel) gives gradient only to the
    # weakest phrase; 'product' gives every phrase gradient weighted by 1/truth.
    t_norm: str = 'min'
    # Weight of each attribute-object binding conjunct.
    binding_loss_weight: float = 1.0
    # Only request relations actually present in the prompt.
    # Example: [("dog", "left_of", "chair")]. Supported: left_of, right_of, above, below.
    relations: List[tuple] = field(default_factory=list)
    token_indices: List[int] = field(default_factory=list)
    # One index list per tracked phrase; membership pools a phrase's own subtokens.
    token_groups: List[List[int]] = field(default_factory=list)
    # Prompt token span (start, end) without the start/end markers or padding.
    text_span: Optional[tuple] = None

    def __post_init__(self):
        if self.height < 64 or self.width != self.height or self.height % 8:
            raise ValueError('Use square images of at least 64 pixels, divisible by 8.')
        if self.n_inference_steps < 1 or self.guidance_scale < 0:
            raise ValueError('Steps must be positive and guidance nonnegative.')
        if self.attn_res < 1 or self.attend_excite_lr < 0 or self.max_iter_to_alter < 0:
            raise ValueError('Invalid attention resolution or guidance update settings.')
        if self.alpha <= 0 or self.spatial_loss_weight < 0:
            raise ValueError('alpha must be positive and spatial weight nonnegative.')
        if self.membership_sharpness <= 0:
            raise ValueError('membership_sharpness must be positive.')
        if self.membership_mode not in ('share', 'relative'):
            raise ValueError("membership_mode must be 'share' or 'relative'.")
        if self.t_norm not in ('min', 'product'):
            raise ValueError("t_norm must be 'min' or 'product'.")
        if self.binding_loss_weight < 0:
            raise ValueError('binding_loss_weight must be nonnegative.')
        if self.text_span is not None:
            start, end = self.text_span
            if not 0 <= start < end:
                raise ValueError('text_span must be an increasing (start, end) pair.')
        self.output_path.mkdir(parents=True, exist_ok=True)


In [ ]:
def soft_truth(membership: torch.Tensor, alpha: float = 10.0) -> torch.Tensor:
    """Graded truth of "the phrase is present": a soft maximum over its membership map."""
    flat = membership.flatten()
    weights = torch.softmax(flat * alpha, dim=0)
    return (weights * flat).sum()


def phrase_membership(attention_map: torch.Tensor, token_group, text_span=None,
                      sharpness: float = 100.0, mode: str = 'share') -> torch.Tensor:
    """Phrase-level spatial membership function with values in [0, 1].

    Share mode sharpens the per-position distribution over the prompt's tokens
    and sums a phrase's token shares. Relative mode instead pools the phrase's raw
    token attention and normalizes by its own spatial peak, allowing overlapping
    attribute/object support. Relative mode ignores sharpness and removes absolute
    strength; neither mode is a calibrated measure of rendered attribute intensity.
    """
    context = attention_map.shape[-1]
    start, end = (0, context) if text_span is None else text_span
    if not 0 <= start < end <= context:
        raise ValueError('Text span lies outside the attention context.')
    group = [idx - start for idx in token_group]
    if any(idx < 0 or idx >= end - start for idx in group):
        raise ValueError('Tracked token lies outside the prompt text span.')
    if mode == 'relative':
        # Use the phrase's own raw attention before the extra token softmax.
        # Co-located attributes and objects can both have high spatial support.
        # This removes absolute strength: even weak/uniform nonzero maps peak at 1.
        return relative_membership(attention_map[..., list(token_group)].sum(-1))
    if mode != 'share':
        raise ValueError(f'Unsupported membership mode: {mode}')
    shares = torch.softmax(attention_map[..., start:end] * sharpness, dim=-1)
    return shares[..., group].sum(-1).clamp(0.0, 1.0)


def relative_membership(membership: torch.Tensor, epsilon: float = 1e-8) -> torch.Tensor:
    """Rescale a membership map so the phrase's own strongest region reads 1.

    Memberships are shares of one per-position distribution over prompt tokens, so
    two phrases can never both be high at the same position: their raw conjunction
    is near zero by construction. Comparing where two phrases are grounded therefore
    has to compare the shape of their support, each normalized to its own peak.
    """
    return membership / membership.max().clamp_min(epsilon)


def soft_left(membership_a: torch.Tensor, membership_b: torch.Tensor) -> torch.Tensor:
    """Graded truth of "a is left of b", from membership centroids."""
    width = membership_a.shape[-1]
    x = torch.linspace(0, 1, width, device=membership_a.device)
    center_a = (membership_a * x).sum() / (membership_a.sum() + 1e-8)
    center_b = (membership_b * x).sum() / (membership_b.sum() + 1e-8)
    return torch.sigmoid(10 * (center_b - center_a))


def soft_under(membership_a: torch.Tensor, membership_b: torch.Tensor) -> torch.Tensor:
    """Graded truth of "a is below b", from membership centroids."""
    height = membership_a.shape[-2]
    y = torch.linspace(0, 1, height, device=membership_a.device).view(height, 1)
    center_a = (membership_a * y).sum() / (membership_a.sum() + 1e-8)
    center_b = (membership_b * y).sum() / (membership_b.sum() + 1e-8)
    return torch.sigmoid(10 * (center_a - center_b))


## Fuzzy objectives

Every tracked phrase is reduced to one spatial membership function, then the
prompt's statements are evaluated on those maps as graded truth values:

1. **Presence** - soft maximum of the membership map; a phrase that stays weakly
   grounded everywhere yields a low truth value.
2. **Spatial relations** - sigmoid comparison of membership centroids, for the
   explicitly requested `left_of`, `right_of`, `above` and `below` constraints only.

Truth values are combined with the Goedel t-norm (`min`) and converted to a loss
by `-log`, so the least satisfied statement drives the latent gradient.


In [ ]:
def compute_fuzzy_loss(avg_attn, token_groups, alpha=10.0, spatial_loss_weight=0.5,
                       relations=(), text_span=None, membership_sharpness=100.0, t_norm='min',
                       binding_loss_weight=1.0, membership_mode='share'):
    maps = [avg_attn[key] for key in ('down_cross', 'mid_cross', 'up_cross')
            if key in avg_attn and avg_attn[key].numel()]
    if not maps or token_groups is None or len(token_groups) == 0:
        raise RuntimeError('No cross-attention maps or tracked tokens for fuzzy guidance.')
    attn_map = torch.stack(maps).mean(0).float()
    groups = normalize_token_groups(token_groups)
    if any(idx < 0 or idx >= attn_map.shape[-1] for group in groups for idx in group):
        raise ValueError('Tracked token is outside the attention context.')
    spatial = math.isqrt(attn_map.shape[1])
    if spatial * spatial != attn_map.shape[1]:
        raise ValueError('Spatial attention must be square.')

    def membership(group):
        return phrase_membership(attn_map, group, text_span, membership_sharpness, membership_mode)

    def conjunction(first, second):
        return torch.minimum(first, second) if t_norm == 'min' else first * second

    truths = torch.stack([soft_truth(membership(group), alpha) for group in groups])
    if t_norm == 'min':
        # Goedel t-norm: only the least satisfied phrase receives gradient.
        loss = -torch.log(truths.min().clamp_min(1e-8))
    elif t_norm == 'product':
        # Product t-norm: every phrase receives gradient, weighted by 1 / truth.
        loss = -torch.log(truths.clamp_min(1e-8)).sum()
    else:
        raise ValueError(f"Unsupported t_norm: {t_norm}")
    for group_a, relation, group_b in relations:
        map_a = membership(group_a).reshape(spatial, spatial)
        map_b = membership(group_b).reshape(spatial, spatial)
        weight = spatial_loss_weight
        if relation == 'left_of':
            truth = soft_left(map_a, map_b)
        elif relation == 'right_of':
            truth = soft_left(map_b, map_a)
        elif relation == 'below':
            truth = soft_under(map_a, map_b)
        elif relation == 'above':
            truth = soft_under(map_b, map_a)
        elif relation == 'bound_to':
            # Attribute binding: the attribute must hold where the object is.
            # A fuzzy AND of the two supports is low exactly when an attribute
            # drifts onto the wrong object, which is the leakage failure mode.
            truth = soft_truth(conjunction(relative_membership(map_a),
                                           relative_membership(map_b)), alpha)
            weight = binding_loss_weight
        else:
            raise ValueError(f'Unsupported relation: {relation}')
        loss = loss - weight * torch.log(truth.clamp_min(1e-8))
    return loss


def normalize_token_groups(token_groups):
    """Accept {phrase: indices}, a list of index groups, or a bare index list."""
    values = list(token_groups.values()) if isinstance(token_groups, dict) else list(token_groups)
    return [[group] if isinstance(group, int) else list(group) for group in values]


def phrase_truth_scores(attn_store, alpha=10.0, membership_sharpness=100.0, token_groups=None,
                        membership_mode=None):
    """Presence truth per tracked phrase, from the accumulated attention maps.

    Recorded in run metadata and used by refinement to tell which phrases the
    base stage left weakly grounded.
    """
    maps = [value for value in attn_store.get_average_attention().values() if value.numel()]
    groups = attn_store.token_groups if token_groups is None else token_groups
    if not maps or not groups:
        return {}
    attn_map = torch.stack(maps).mean(0).float()
    span = getattr(attn_store, 'text_span', None)
    mode = membership_mode if membership_mode is not None else getattr(attn_store, 'membership_mode', 'share')
    return {phrase: soft_truth(phrase_membership(attn_map, group, span, membership_sharpness, mode),
                               alpha).item()
            for phrase, group in groups.items()}


In [ ]:
# Evaluation on CPU keeps GPU memory available for gradient-enabled SDXL.
clip_device = torch.device('cpu')
clip_model, clip_preprocess = clip.load('ViT-B/32', device=clip_device)
clip_model.eval().requires_grad_(False)


In [ ]:
# Attention collection is explicitly bounded by each UNet forward.
# No layer counter: self-attention is handled by Diffusers' efficient processors.


In [ ]:
class AttentionStore:
    def __init__(self, attn_res=16):
        self.attn_res = attn_res
        self.token_groups = {}
        # Prompt token span used to turn attention into membership; None means the
        # whole context, including markers and padding.
        self.text_span = None
        self.reset()

    def begin_forward(self, keep_grad=False, cfg=False):
        self.step_store = {}
        self.keep_grad = keep_grad
        self.cfg = cfg
        self.active = True

    def forward(self, attn, is_cross, place_in_unet):
        if not self.active or not is_cross:
            return attn
        size = math.isqrt(attn.shape[1])
        if size * size != attn.shape[1]:
            raise ValueError('Only square attention maps are supported.')
        averaged = attn.float().mean(0, keepdim=True)
        resized = F.interpolate(averaged.reshape(1, size, size, -1).permute(0, 3, 1, 2),
                                size=(self.attn_res, self.attn_res), mode='bilinear',
                                align_corners=False)
        resized = resized.permute(0, 2, 3, 1).reshape(1, self.attn_res ** 2, -1)
        key = f'{place_in_unet}_cross'
        self.step_store.setdefault(key, []).append(resized if self.keep_grad else resized.detach())
        return attn

    def get_current_step_attention(self):
        return {key: torch.stack(values).mean(0) for key, values in self.step_store.items() if values}

    def end_forward(self, accumulate=True):
        if accumulate:
            for key, values in self.get_current_step_attention().items():
                values = values.detach().cpu()
                self.attention_store[key] = self.attention_store.get(key, torch.zeros_like(values)) + values
                self.counts[key] = self.counts.get(key, 0) + 1
        self.step_store = {}
        self.active = False

    def get_average_attention(self):
        return {key: value / self.counts[key] for key, value in self.attention_store.items()}

    def reset(self):
        self.attention_store = {}
        self.counts = {}
        self.begin_forward()
        self.guidance_diagnostics = []


## Attention processing

Only cross-attention at up to 32 x 32 spatial resolution is intercepted.
Higher resolutions and all self-attention retain Diffusers' efficient processors.
Maps are resized to a common grid before membership is computed. Guidance maps
preserve the graph; accumulated maps are detached and averaged on CPU. CFG's
conditional batch is selected after separating the batch and attention-head
dimensions, so the negative branch never enters a membership function.


In [ ]:
class AttendExciteAttnProcessor:
    def __init__(self, attnstore=None, place_in_unet='mid', fallback=None):
        self.attnstore = attnstore
        self.place_in_unet = place_in_unet
        self.fallback = fallback

    def modify_attention(self, probabilities, batch_size, heads):
        if self.attnstore is not None and self.attnstore.active:
            shaped = probabilities.reshape(batch_size, heads, *probabilities.shape[1:])
            conditional = shaped[-1] if self.attnstore.cfg else shaped[0]
            self.attnstore.forward(conditional, True, self.place_in_unet)
        return probabilities

    def __call__(self, attn, hidden_states, encoder_hidden_states=None,
                 attention_mask=None, temb=None, **kwargs):
        spatial_tokens = hidden_states.shape[1] if hidden_states.ndim == 3 else math.prod(hidden_states.shape[-2:])
        if encoder_hidden_states is None or (self.attnstore is not None and spatial_tokens > 32 ** 2):
            return self.fallback(attn, hidden_states, encoder_hidden_states=encoder_hidden_states,
                                 attention_mask=attention_mask, temb=temb)
        residual = hidden_states
        if attn.spatial_norm is not None:
            hidden_states = attn.spatial_norm(hidden_states, temb)
        input_ndim = hidden_states.ndim
        if input_ndim == 4:
            batch_size, channel, height, width = hidden_states.shape
            hidden_states = hidden_states.view(batch_size, channel, height * width).transpose(1, 2)
        batch_size = hidden_states.shape[0]
        if attention_mask is not None:
            attention_mask = attn.prepare_attention_mask(attention_mask, encoder_hidden_states.shape[1], batch_size)
        if attn.group_norm is not None:
            hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)
        if attn.norm_cross:
            encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)
        query = attn.head_to_batch_dim(attn.to_q(hidden_states))
        key = attn.head_to_batch_dim(attn.to_k(encoder_hidden_states))
        value = attn.head_to_batch_dim(attn.to_v(encoder_hidden_states))
        probabilities = attn.get_attention_scores(query, key, attention_mask)
        probabilities = self.modify_attention(probabilities, batch_size, attn.heads)
        hidden_states = attn.batch_to_head_dim(torch.bmm(probabilities, value))
        hidden_states = attn.to_out[1](attn.to_out[0](hidden_states))
        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(batch_size, channel, height, width)
        if attn.residual_connection:
            hidden_states = hidden_states + residual
        return hidden_states / attn.rescale_output_factor


In [ ]:
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from diffusers.pipelines.stable_diffusion_xl.pipeline_output import StableDiffusionXLPipelineOutput


In [ ]:
class FuzzyAttendExciteSDXLPipeline(StableDiffusionXLPipeline):
    """Single-prompt, square-image SDXL pipeline with explicit latent guidance."""
    def register_attention_control(self, controller):
        if not hasattr(self, '_original_attention_processors'):
            self._original_attention_processors = dict(self.unet.attn_processors)
        processors = {}
        for name, original in self._original_attention_processors.items():
            place = 'mid' if name.startswith('mid') else ('up' if name.startswith('up') else 'down')
            processors[name] = (AttendExciteAttnProcessor(controller, place, original)
                                if '.attn2.' in name else original)
        self.unet.set_attn_processor(processors)

    def _update_latents_with_fuzzy_loss(self, latents, attention_store, token_indices,
                                       cfg, timestep, prompt_embeds, added_cond_kwargs,
                                       relations=(), step_index=0):
        with torch.enable_grad():
            # Float32 leaf prevents small fp16 latent updates from rounding away.
            leaf = latents.detach().float().requires_grad_(True)
            attention_store.begin_forward(keep_grad=True)
            try:
                model_input = self.scheduler.scale_model_input(leaf.to(prompt_embeds.dtype), timestep)
                self.unet(model_input, timestep, encoder_hidden_states=prompt_embeds,
                          added_cond_kwargs=added_cond_kwargs, return_dict=False)
                loss = compute_fuzzy_loss(attention_store.get_current_step_attention(),
                                          cfg.token_groups or token_indices, cfg.alpha,
                                          cfg.spatial_loss_weight, relations,
                                          cfg.text_span, cfg.membership_sharpness,
                                          getattr(cfg, 't_norm', 'min'),
                                          getattr(cfg, 'binding_loss_weight', 1.0),
                                          getattr(cfg, 'membership_mode', 'share'))
                if not loss.requires_grad:
                    raise RuntimeError('Fuzzy loss is disconnected from the latent graph.')
                gradient = torch.autograd.grad(loss, leaf)[0]
                if not torch.isfinite(loss) or not torch.isfinite(gradient).all():
                    raise FloatingPointError('Nonfinite fuzzy loss or gradient; reduce guidance settings.')
                rms = gradient.square().mean().sqrt()
                if rms.item() == 0:
                    raise RuntimeError('Fuzzy latent gradient is zero.')
                # Preserve gradient magnitude: normalization amplified tiny gradients into disruptive updates.
                rate = cfg.attend_excite_lr * math.sqrt(max(0.0, 1 - step_index / cfg.n_inference_steps))
                delta_tensor = rate * gradient
                delta_rms = delta_tensor.square().mean().sqrt()
                # Clip large updates, but NEVER amplify small ones.
                ceiling = (0.01 / delta_rms.clamp_min(1e-12)).clamp(max=1.0)
                delta_tensor = delta_tensor * ceiling
                updated = (leaf - delta_tensor).detach()
                delta = (updated.float() - latents.float()).square().mean().sqrt().item()
                latent_rms = latents.float().square().mean().sqrt().item()
                # update_ratio is the diagnostic to read first: it says whether the
                # update is large enough to matter at this latent scale, and
                # 'clipped' says whether the absolute ceiling is what limited it.
                attention_store.guidance_diagnostics.append(
                    {'step': step_index, 'loss': loss.item(), 'gradient_rms': rms.item(), 'update_rms': delta,
                     'learning_rate': rate, 'clipped': bool(ceiling.item() < 1.0),
                     'latent_rms': latent_rms, 'update_ratio': delta / max(latent_rms, 1e-12)})
                return updated
            finally:
                attention_store.end_forward(accumulate=False)

    @torch.no_grad()
    def __call__(self, prompt, token_indices=None, attention_store=None, height=768, width=768,
                 num_inference_steps=50, guidance_scale=9.5, generator=None, output_type='pil',
                 cfg=None, negative_prompt=None, relations=(), return_dict=True):
        if not isinstance(prompt, str):
            raise ValueError('This notebook supports one prompt/image per call.')
        cfg = cfg or RunConfig(prompt=prompt, height=height, width=width,
                               n_inference_steps=num_inference_steps, guidance_scale=guidance_scale)
        if height != width or height % 8 or height < 64 or num_inference_steps < 1 or guidance_scale < 0:
            raise ValueError('Use positive steps, nonnegative guidance and square dimensions divisible by 8.')
        if output_type not in ('pil', 'np', 'latent'):
            raise ValueError('output_type must be pil, np or latent.')
        token_indices = token_indices or []
        do_cfg = guidance_scale > 1.0
        exec_device = self._execution_device
        if attention_store is not None:
            attention_store.reset()
        positive, negative, pooled, negative_pooled = self.encode_prompt(
            prompt=prompt, device=exec_device, num_images_per_prompt=1,
            do_classifier_free_guidance=do_cfg, negative_prompt=negative_prompt or '')
        projection_dim = (self.text_encoder_2.config.projection_dim if self.text_encoder_2 is not None
                          else pooled.shape[-1])
        time_ids = self._get_add_time_ids((height, width), (0, 0), (height, width),
                                        dtype=positive.dtype, text_encoder_projection_dim=projection_dim).to(exec_device)
        embeddings = torch.cat([negative, positive]) if do_cfg else positive
        pooled_embeddings = torch.cat([negative_pooled, pooled]) if do_cfg else pooled
        full_time_ids = torch.cat([time_ids, time_ids]) if do_cfg else time_ids
        self.scheduler.set_timesteps(num_inference_steps, device=exec_device)
        latents = self.prepare_latents(1, self.unet.config.in_channels, height, width,
                                       positive.dtype, exec_device, generator, None).float()
        extra_step_kwargs = self.prepare_extra_step_kwargs(generator, eta=0.0)
        try:
            for i, timestep in enumerate(self.scheduler.timesteps):
                if attention_store is not None and token_indices and i < cfg.max_iter_to_alter and cfg.attend_excite_lr > 0:
                    latents = self._update_latents_with_fuzzy_loss(
                        latents, attention_store, token_indices, cfg, timestep, positive,
                        {'text_embeds': pooled, 'time_ids': time_ids}, relations, i)
                # Predict noise from UPDATED latents at the SAME timestep.
                model_input = torch.cat([latents, latents]) if do_cfg else latents
                model_input = self.scheduler.scale_model_input(model_input, timestep)
                if attention_store is not None:
                    attention_store.begin_forward(cfg=do_cfg)
                noise = self.unet(model_input.to(embeddings.dtype), timestep, encoder_hidden_states=embeddings,
                                  added_cond_kwargs={'text_embeds': pooled_embeddings, 'time_ids': full_time_ids},
                                  return_dict=False)[0]
                if attention_store is not None:
                    attention_store.end_forward()
                if do_cfg:
                    unconditional, conditional = noise.chunk(2)
                    noise = unconditional + guidance_scale * (conditional - unconditional)
                latents = self.scheduler.step(noise.float(), timestep, latents, **extra_step_kwargs).prev_sample
            if output_type == 'latent':
                images = latents
            else:
                # Honor SDXL's VAE precision and inherited image postprocessing.
                original_dtype = self.vae.dtype
                needs_upcast = original_dtype == torch.float16 and self.vae.config.force_upcast
                if needs_upcast:
                    self.upcast_vae()
                decode_dtype = next(self.vae.post_quant_conv.parameters()).dtype
                decoded = self.vae.decode(latents.to(decode_dtype) / self.vae.config.scaling_factor,
                                          return_dict=False)[0]
                if needs_upcast:
                    self.vae.to(dtype=original_dtype)
                images = self.image_processor.postprocess(decoded, output_type=output_type)
            return StableDiffusionXLPipelineOutput(images=images) if return_dict else (images,)
        finally:
            if attention_store is not None:
                attention_store.end_forward(accumulate=False)
            self.maybe_free_model_hooks()


In [ ]:
def get_token_groups(tokenizer, prompt, words_to_track):
    token_ids = tokenizer.encode(prompt)
    if len(token_ids) > tokenizer.model_max_length:
        raise ValueError('Prompt exceeds the tokenizer context; shorten it instead of silently truncating.')
    groups = {}
    for word in words_to_track:
        word_ids = tokenizer.encode(word, add_special_tokens=False)
        if not word_ids:
            raise ValueError('Tracked phrases must not be empty.')
        indices = []
        for start in range(1, len(token_ids) - len(word_ids)):
            if token_ids[start:start + len(word_ids)] == word_ids:
                # Do not match a suffix of a longer BPE word.
                previous = tokenizer.convert_ids_to_tokens([token_ids[start - 1]])[0]
                if start == 1 or previous.endswith('</w>'):
                    indices.extend(range(start, start + len(word_ids)))
        if not indices:
            raise ValueError(f'Tracked phrase {word!r} was not found in the prompt.')
        groups[word] = sorted(set(indices))
    return groups


def get_token_indices(tokenizer, prompt, words_to_track):
    groups = get_token_groups(tokenizer, prompt, words_to_track)
    return sorted({idx for indices in groups.values() for idx in indices})


In [ ]:
# Direct prompt similarity avoids adding unrelated "dirty", "small", etc. templates.
imagenet_templates = ['{}']


In [ ]:
def get_embedding_for_prompt(model, prompt, templates):
    with torch.no_grad():
        tokens = clip.tokenize([template.format(prompt) for template in templates], truncate=True).to(clip_device)
        embeddings = F.normalize(model.encode_text(tokens).float(), dim=-1)
        return F.normalize(embeddings.mean(0), dim=0)


In [ ]:
@torch.no_grad()
def clip_prompt_similarities(image: Image.Image, prompt: str):
    """Compute CLIP similarity between image and prompt parts."""
    if " and " in prompt:
        parts = prompt.split(" and ")
        p1    = parts[0].strip()
        p2    = " and ".join(parts[1:]).strip()
    else:
        p1 = p2 = prompt

    image_tensor = clip_preprocess(image).unsqueeze(0).to(clip_device)
    image_feat   = clip_model.encode_image(image_tensor)
    image_feat  /= image_feat.norm(dim=-1, keepdim=True)
    image_feat   = image_feat.float()

    full_feat = get_embedding_for_prompt(clip_model, prompt, imagenet_templates)
    p1_feat   = get_embedding_for_prompt(clip_model, p1,     imagenet_templates)
    p2_feat   = get_embedding_for_prompt(clip_model, p2,     imagenet_templates)

    return {
        "full_text":   (image_feat @ full_feat.T).item(),
        "first_half":  (image_feat @ p1_feat.T).item(),
        "second_half": (image_feat @ p2_feat.T).item(),
        "min_half":    min((image_feat @ p1_feat.T).item(),
                           (image_feat @ p2_feat.T).item()),
    }


In [ ]:
def generate(prompt, words_to_track, seed=84, num_steps=50, guidance=9.5, height=768, width=768,
             max_iter_to_alter=30, attend_excite_lr=0.2, alpha=10.0, spatial_loss_weight=0.5,
             negative_prompt='', relations=(), attn_res=16, membership_sharpness=100.0,
             t_norm='min', binding_loss_weight=1.0, pipeline=None, membership_mode='share'):
    pipeline = pipeline if pipeline is not None else model
    cfg = RunConfig(prompt=prompt, seeds=[seed], n_inference_steps=num_steps, guidance_scale=guidance,
                    height=height, width=width, max_iter_to_alter=max_iter_to_alter,
                    attend_excite_lr=attend_excite_lr, alpha=alpha,
                    spatial_loss_weight=spatial_loss_weight, relations=list(relations), attn_res=attn_res,
                    membership_sharpness=membership_sharpness, t_norm=t_norm,
                    binding_loss_weight=binding_loss_weight, membership_mode=membership_mode)
    groups = get_token_groups(pipeline.tokenizer, prompt, words_to_track)
    second_groups = get_token_groups(pipeline.tokenizer_2, prompt, words_to_track)
    if groups != second_groups:
        raise ValueError('SDXL tokenizers disagree on tracked phrase positions; use an aligned prompt.')
    indices = sorted({idx for group in groups.values() for idx in group})
    if not indices:
        raise ValueError('Provide at least one tracked word/phrase.')
    token_ids = pipeline.tokenizer.encode(prompt)
    if len(token_ids) != len(pipeline.tokenizer_2.encode(prompt)):
        raise ValueError('SDXL tokenizers disagree on prompt length; use an aligned prompt.')
    resolved_relations = []
    for word_a, relation, word_b in relations:
        if word_a not in groups or word_b not in groups:
            raise ValueError('Both relation operands must be tracked phrases.')
        if relation not in ('left_of', 'right_of', 'above', 'below', 'bound_to'):
            raise ValueError(f'Unsupported relation: {relation}')
        resolved_relations.append((groups[word_a], relation, groups[word_b]))
    cfg.token_indices = indices
    # Membership is a phrase's share of the prompt's own tokens, so the start/end
    # markers and the padding tail are excluded from the distribution.
    cfg.text_span = (1, len(token_ids) - 1)
    cfg.token_groups = [groups[word] for word in words_to_track]
    store = AttentionStore(attn_res=cfg.attn_res)
    store.token_groups = groups
    store.text_span = cfg.text_span
    store.membership_mode = cfg.membership_mode
    pipeline.register_attention_control(store)
    generator = torch.Generator(device='cpu').manual_seed(seed)
    result = pipeline(prompt=prompt, token_indices=indices, attention_store=store, height=height, width=width,
                      num_inference_steps=num_steps, guidance_scale=guidance, generator=generator, cfg=cfg,
                      negative_prompt=negative_prompt, relations=resolved_relations)
    image = result.images[0]
    scores = clip_prompt_similarities(image, prompt)
    print('CLIP similarity:', scores)
    print('Phrase presence truth:', phrase_truth_scores(store, cfg.alpha, cfg.membership_sharpness))
    if store.guidance_diagnostics:
        print('First guidance update:', store.guidance_diagnostics[0])
    return image, scores, store


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Kaggle notebook settings before loading SDXL.')
device = torch.device('cuda:0')
torch_dtype = torch.float16
print('Using:', torch.cuda.get_device_name(0), 'GPU count:', torch.cuda.device_count())
print('This notebook offloads models to CPU and uses cuda:0 only.')


In [ ]:
# Load VAE to CPU first; enable_model_cpu_offload will handle device placement
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch_dtype,
)

In [ ]:
model = FuzzyAttendExciteSDXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae            = vae,
    torch_dtype    = torch_dtype,
    variant        = "fp16" if torch.cuda.is_available() else None,
    use_safetensors= True,
)


In [ ]:
model.unet.eval().requires_grad_(False)
model.vae.eval().requires_grad_(False)
model.text_encoder.eval().requires_grad_(False)
model.text_encoder_2.eval().requires_grad_(False)
model.unet.enable_gradient_checkpointing()
model.enable_model_cpu_offload(gpu_id=0)
model.enable_vae_tiling()
model.scheduler = DPMSolverMultistepScheduler.from_config(
    model.scheduler.config, algorithm_type='dpmsolver++', use_karras_sigmas=True)
print('SDXL base loaded with CPU offload, efficient self-attention and gradient checkpointing.')


# Stage 2: spatially guided attention refinement

Reuse the same phrase-level membership functions as soft spatial masks, and let
the image-to-image refiner strengthen conditional cross-attention inside each
phrase's region. The correction is scaled by the phrase's membership deficit
`1 - truth`, so residual omissions are corrected and already grounded phrases
are left alone.


In [ ]:
def build_phrase_membership_masks(attn_store, device, latent_size=128, sharpness=100.0,
                                  token_groups=None, membership_mode=None):
    """Phrase membership masks for refinement, resized to the refiner's latent grid."""
    maps = [value for value in attn_store.get_average_attention().values() if value.numel()]
    groups = attn_store.token_groups if token_groups is None else token_groups
    if not maps or not groups:
        return {}
    attn_map = torch.stack(maps).mean(0).float()
    span = getattr(attn_store, 'text_span', None)
    mode = membership_mode if membership_mode is not None else getattr(attn_store, 'membership_mode', 'share')
    masks = {}
    for phrase, group in groups.items():
        membership = phrase_membership(attn_map, group, span, sharpness, mode)
        spatial = math.isqrt(membership.numel())
        if spatial * spatial != membership.numel():
            raise ValueError('Membership masks require square attention.')
        if membership.max() <= 1e-8:
            continue
        # Relative membership puts the phrase's strongest region at 1 for reweighting.
        relative = relative_membership(membership).reshape(1, 1, spatial, spatial)
        mask = F.interpolate(relative, size=(latent_size, latent_size), mode='bilinear',
                             align_corners=False)[0, 0]
        masks[phrase] = mask.clamp(0, 1).to(device)
    return masks


In [ ]:
class PredicateRefinerProcessor(AttendExciteAttnProcessor):
    """Spatially guided attention refinement.

    A phrase's membership mask strengthens conditional cross-attention inside the
    phrase's region and discourages it outside. The per-token weight in [0, 1]
    scales the whole correction, so a phrase the base stage already grounded is
    left untouched while a residual omission receives the full correction.
    """
    def __init__(self, masks, strength=2.5, weights=None, cfg=True, fallback=None):
        super().__init__(fallback=fallback)
        self.masks = masks
        self.strength = strength
        self.weights = {idx: 1.0 for idx in masks} if weights is None else dict(weights)
        missing = sorted(set(masks) - set(self.weights))
        if missing:
            raise ValueError(f'Missing refinement weights for token indices: {missing}')
        self.cfg = cfg

    def modify_attention(self, probabilities, batch_size, heads):
        if batch_size != (2 if self.cfg else 1):
            raise ValueError('Refinement supports one image per call.')
        spatial = math.isqrt(probabilities.shape[1])
        if spatial * spatial != probabilities.shape[1]:
            raise ValueError('Refinement attention must be square.')
        shaped = probabilities.reshape(batch_size, heads, *probabilities.shape[1:])
        # CFG ordering is [unconditional, conditional]; leave the first branch intact.
        conditional = shaped[-1].clone()
        for idx, full_mask in self.masks.items():
            if idx >= conditional.shape[-1]:
                raise ValueError('Refiner token index is outside the prompt context.')
            weight = self.weights[idx]
            if weight <= 0:
                continue
            mask = F.interpolate(full_mask.to(probabilities.device).view(1, 1, *full_mask.shape),
                                 size=(spatial, spatial), mode='bilinear', align_corners=False).reshape(1, -1)
            correction = (1 + self.strength * mask) * (1 - 0.7 * (1 - mask))
            factor = 1 + weight * (correction - 1)
            conditional[:, :, idx] *= factor.to(conditional.dtype)
        conditional = conditional / conditional.sum(-1, keepdim=True).clamp_min(1e-8)
        shaped = torch.cat([shaped[:1], conditional.unsqueeze(0)]) if self.cfg else conditional.unsqueeze(0)
        return shaped.reshape_as(probabilities)


In [ ]:
from diffusers import StableDiffusionXLImg2ImgPipeline

def load_refiner(device):
    refiner = StableDiffusionXLImg2ImgPipeline.from_pretrained(
        'stabilityai/stable-diffusion-xl-refiner-1.0', torch_dtype=torch.float16,
        variant='fp16', use_safetensors=True)
    refiner.enable_model_cpu_offload(gpu_id=device.index or 0)
    refiner.enable_vae_tiling()
    refiner.unet.eval().requires_grad_(False)
    refiner.scheduler = DPMSolverMultistepScheduler.from_config(
        refiner.scheduler.config, algorithm_type='dpmsolver++', use_karras_sigmas=True)
    return refiner


In [ ]:
def attach_predicate_control(refiner, masks, strength=2.5, weights=None, cfg=True):
    if not hasattr(refiner, '_original_attention_processors'):
        refiner._original_attention_processors = dict(refiner.unet.attn_processors)
    processors = {name: (PredicateRefinerProcessor(masks, strength, weights, cfg, original)
                         if '.attn2.' in name else original)
                  for name, original in refiner._original_attention_processors.items()}
    refiner.unet.set_attn_processor(processors)


In [ ]:
def refine_with_predicate(image, prompt, attn_store, seed=84, negative_prompt='',
                          refiner=None, strength=0.4, num_steps=30, guidance=9.5,
                          predicate_strength=2.5, alpha=10.0, membership_sharpness=100.0,
                          membership_mode=None):
    if not 0 < strength <= 1 or int(num_steps * strength) < 1:
        raise ValueError('Refinement strength/steps must yield at least one denoising step.')
    groups = dict(attn_store.token_groups)
    if not groups:
        raise ValueError('Refinement needs the tracked phrase groups recorded by generate().')
    phrase_masks = build_phrase_membership_masks(attn_store, torch.device('cpu'), image.width // 8,
                                                 membership_sharpness, groups, membership_mode)
    if not phrase_masks:
        raise RuntimeError('No nonuniform attention masks were captured for refinement.')
    truths = phrase_truth_scores(attn_store, alpha, membership_sharpness, groups, membership_mode)
    own_refiner = refiner is None
    refiner = refiner if refiner is not None else load_refiner(device)
    try:
        # The refiner conditions on tokenizer_2, not the base model CLIP-L tokenizer.
        target_groups = get_token_groups(refiner.tokenizer_2, prompt, list(groups))
        masks, weights = {}, {}
        for phrase, phrase_mask in phrase_masks.items():
            # Residual omissions get the full correction; grounded phrases get none.
            deficit = min(1.0, max(0.0, 1.0 - truths.get(phrase, 0.0)))
            for idx in target_groups[phrase]:
                masks[idx] = phrase_mask
                weights[idx] = deficit
        missing = sorted(set(groups) - set(phrase_masks))
        if missing:
            raise RuntimeError(f'No usable mask for tracked phrases: {missing}')
        attach_predicate_control(refiner, masks, predicate_strength, weights, cfg=guidance > 1)
        print('Refinement correction weights:', weights)
        refined = refiner(prompt=prompt, image=image, negative_prompt=negative_prompt,
                          strength=strength, num_inference_steps=num_steps, guidance_scale=guidance,
                          generator=torch.Generator(device='cpu').manual_seed(seed)).images[0]
        return refined
    finally:
        if hasattr(refiner, '_original_attention_processors'):
            refiner.unet.set_attn_processor(dict(refiner._original_attention_processors))
        refiner.maybe_free_model_hooks()
        if own_refiner:
            del refiner
            gc.collect()
            torch.cuda.empty_cache()


In [ ]:
from IPython.display import display

# First run a short end-to-end GPU check; switch to False for the full settings.
SMOKE_TEST = not globals().get('FULL_RUN', False)
options = globals().get('RUN_OPTIONS', {})
prompt = options.get('prompt', 'A very fast car')
words_to_track = options.get('words', ['fast', 'car'])
seeds = [options.get('seed', 142)]
negative_prompt = ('cartoon, anime, illustration, painting, drawing, sketch, 3d render, cgi, '
                   'toy-like, plastic texture, low quality, blurry, distorted, text, watermark')
# Do not invent spatial constraints for attributes like "fast".
relations = options.get('relations', [])
run = RunConfig(prompt=prompt, seeds=seeds, height=512 if SMOKE_TEST else 768,
                width=512 if SMOKE_TEST else 768, n_inference_steps=4 if SMOKE_TEST else 50,
                max_iter_to_alter=2 if SMOKE_TEST else 30, relations=relations,
                attend_excite_lr=options.get('attend_excite_lr', 0.2),
                membership_sharpness=options.get('membership_sharpness', 100.0),
                membership_mode=options.get('membership_mode', 'share'),
                t_norm=options.get('t_norm', 'min'),
                binding_loss_weight=options.get('binding_loss_weight', 1.0),
                **({'output_path': Path(options['output'])} if options.get('output') else {}))
records = []
pending = []
for seed in seeds:
    image, scores, store = generate(
        prompt, words_to_track, seed=seed, num_steps=run.n_inference_steps,
        guidance=run.guidance_scale, height=run.height, width=run.width,
        max_iter_to_alter=run.max_iter_to_alter, attend_excite_lr=run.attend_excite_lr,
        alpha=run.alpha, spatial_loss_weight=run.spatial_loss_weight,
        negative_prompt=negative_prompt, relations=run.relations, attn_res=run.attn_res,
        membership_sharpness=run.membership_sharpness, membership_mode=run.membership_mode,
        t_norm=run.t_norm, binding_loss_weight=run.binding_loss_weight, pipeline=model)
    stage1_path = run.output_path / f'seed_{seed}_stage1.png'
    image.save(stage1_path)  # Persist success BEFORE loading the refiner.
    record = {'seed': seed, 'prompt': prompt, 'words_to_track': words_to_track,
              'negative_prompt': negative_prompt, 'config': {**asdict(run), 'output_path': str(run.output_path),
                         'token_indices': sorted({idx for group in store.token_groups.values() for idx in group}),
                         'token_groups': store.token_groups, 'text_span': store.text_span},
              'stage1': str(stage1_path), 'stage1_clip': scores,
              'stage1_phrase_truth': phrase_truth_scores(store, run.alpha, run.membership_sharpness),
              'truth_note': 'Attention-derived objective, not visual accuracy or attribute intensity.',
              'guidance_diagnostics': store.guidance_diagnostics, 'refinement_status': 'pending',
              'versions': {'torch': torch.__version__, 'diffusers': __import__('diffusers').__version__,
                           'transformers': __import__('transformers').__version__}}
    records.append(record)
    (run.output_path / f'seed_{seed}_metadata.json').write_text(json.dumps(record, indent=2), encoding='utf-8')
    pending.append((seed, image, store, record))

# Stage all base outputs, then unload base weights before bringing up refinement.
model.remove_all_hooks()
model.to('cpu')
del model, vae
gc.collect()
torch.cuda.empty_cache()
refiner = None
try:
    refiner = load_refiner(device)
    for seed, image, store, record in pending:
        try:
            refined = refine_with_predicate(image, prompt, store, seed=seed,
                negative_prompt=negative_prompt, refiner=refiner,
                alpha=run.alpha, membership_sharpness=run.membership_sharpness,
                membership_mode=run.membership_mode,
                num_steps=8 if SMOKE_TEST else 30)
            stage2_path = run.output_path / f'seed_{seed}_stage2_refined.png'
            refined.save(stage2_path)
            record.update(stage2=str(stage2_path), stage2_clip=clip_prompt_similarities(refined, prompt),
                          refinement_status='complete')
            print('Stage 1 phrase truth:', record['stage1_phrase_truth'])
            display(refined)
        except Exception as error:
            record.update(refinement_status='failed', refinement_error=f'{type(error).__name__}: {error}')
            raise
        finally:
            (run.output_path / f'seed_{seed}_metadata.json').write_text(json.dumps(record, indent=2), encoding='utf-8')
except Exception as error:
    for record in records:
        if record['refinement_status'] == 'pending':
            record.update(refinement_status='failed', refinement_error=f'{type(error).__name__}: {error}')
            (run.output_path / f"seed_{record['seed']}_metadata.json").write_text(json.dumps(record, indent=2), encoding='utf-8')
    raise
finally:
    if refiner is not None:
        refiner.remove_all_hooks()
        refiner.to('cpu')
        del refiner
    gc.collect()
    torch.cuda.empty_cache()
print('Images and metadata saved to:', run.output_path.resolve())
